In [252]:
#This was done with the help of AI. 
# It was suggested to me to_parquet to save the cleaned data as a parquet file.
# I have done this and it is now saved in the data folder as cps_income_clean.parquet.
#Suposedly this is a more efficient way to store data than csv files. Saves space and faster...??? 

Import necessary libraries ('usually at the top')

In [253]:
from pathlib import Path
import sys
import os
import sqlite3

# Project root folder
PROJECT_ROOT = Path.cwd().parent

# Data and database locations
DB_PATH = PROJECT_ROOT / "crimedatacsv_new.db"
DATA_PATH = PROJECT_ROOT / "data"

print("Project:", PROJECT_ROOT)
print("Database:", DB_PATH)

conn = sqlite3.connect(DB_PATH)
print("Connection opened:", conn)

print("Python:")
print(sys.executable)

print("\nWorking directory:")
print(os.getcwd())

print("\nDatabase opened successfully")

print("\nDatabase exists:", DB_PATH.exists())

Project: c:\Users\bperf\OneDrive\Desktop\UBIAnalysis
Database: c:\Users\bperf\OneDrive\Desktop\UBIAnalysis\crimedatacsv_new.db
Connection opened: <sqlite3.Connection object at 0x0000020260D24400>
Python:
c:\Users\bperf\OneDrive\Desktop\UBIAnalysis\.venv\Scripts\python.exe

Working directory:
c:\Users\bperf\OneDrive\Desktop\UBIAnalysis\notebooks

Database opened successfully

Database exists: True


In [254]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import matplotlib.ticker as mticker
import os
import folium


print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.3
NumPy: 2.4.5


# **AI assisted-checked by AI**
# **AI suggested-suggested a query not considered...also proper form and function...**

Establishing a connection with sqlite3: means openning a connection to the sqlite database (db) file, and creates a db container for csv(s).


Due to memory constraints, because of the large csv(s); I got a sample of 1500 rows from each of the orginal (large) datasets.

crime_sample = crime_df.sample(1500, random_state=42)

crime_sample     #Creates a new DataFrame called crime_sample.
= crime_df       #This is your original full crime dataset.
.sample(1500,    #This is a pandas function that randomly selects rows.
random_state=42) #This makes the random selection repeatable. Otherwise the same random sample would
                  not be repeatable; you would get a different random sample each time.


   crime_sample          #csv variable df_name
   .to_sql               #a pandas function that creates the table(s) in the db.
(  
    "crime_sample",      #name of table inside database
    conn,                #connection to sql dababase
    index=False,         #gets rid of the indexing
    if_exists="replace"  #if another table exist in the db with the same name replace it with this
                            one.
)


Load Dataset (CSV) into a dataframe:  df = pd.read_csv("your_file.csv") - The csv is being 'read' into a pandas DataFrame (df).


In [255]:
la_crime = pd.read_csv(DATA_PATH /"la_crime_1500.csv")
local_crime = pd.read_csv(DATA_PATH /"local_crime_1500.csv")
cps_income = pd.read_csv(DATA_PATH /"cps_income_1500.csv")
cps_income.shape

(1500, 16)

Writes the df(s) into a table(s):  csv_variable_name.to_sql("table name", conn, index=False, if_exists="replace")- — 'it takes the DataFrame data and creates a SQLite table containing that data.'

In [256]:
la_crime.to_sql(        #csv variable df_name
    "la_crime",         #name of table inside database
    conn,               #connection to sql dababase
    index=False,        #gets rid of the indexing
    if_exists="replace" #if another table exist in the db with the same name replace it with this one.
)

local_crime.to_sql(
    "local_crime",
    conn,
    index=False,
    if_exists="replace"
)

cps_income.to_sql(
    "cps_income",
    conn,
    index=False,
    if_exists="replace"
)

# conn.close()  #'sqlite3.connect() → starts the phone call to_sql() / SQL queries → you communicate through the call conn.close() → hangs up the phone'

print("Database created successfully!") #prints if ran properly

Database created successfully!


To see the results of the connection, this query shows all the tables that exist in the db. Using pandas it returns a table(s) df name.

In [257]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

,name
0,ubi_subset
1,la_crime
2,local_crime
3,cps_income


Cleaning ('Wrangling Data'):  removing duplicates, overly excessive columns with nulls, datetime inconsistencies
and replacing table codes with title schema...

In [258]:
# # la_crime.isnull().sum()
# # la_crime = la_crime.fillna(0)
# # la_crime.info()
# la_crime.shape
la_crime.columns

Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT',
       'LON'],
      dtype='str')

In [259]:
# local_crime.isnull().sum()
# local_crime = local_crime.fillna(0)
# local_crime.info()
local_crime.columns
local_crime.shape

(1500, 16)

In [260]:
# cps_income.isnull().sum()
# cps_income.info()
cps_income.columns
cps_income.shape

(1500, 16)

#Using .map to change the codes for the sexes to words: 1 - for males, and 2 - females, making the chart more readable. 

In [261]:
sex_map = {
    1: 'Male',
    2: 'Female'
}

cps_income['SEX'] = cps_income['SEX'].map(sex_map)
cps_income.head()


,YEAR,REGION,STATEFIP,COUNTY,METRO,ASECWT,AGE,SEX,RACE,OCC,UHRSWORK1,EDUC,SCHLCOLL,INCWAGE,OFFTOTVAL,POVERTY
0,2024,33,48,0,2,1898.77,80,Male,100,0,999,40,0,0,91488,23
1,2024,31,13,0,3,3187.62,64,Female,100,800,40,111,0,161000,294060,23
2,2024,12,36,0,4,2597.91,17,Female,100,0,999,60,1,0,55002,23
3,2024,21,17,17119,3,1810.53,25,Female,100,0,999,111,5,30000,43201,21
4,2024,31,37,37001,4,11338.34,45,Male,100,6600,45,71,5,55000,65000,23


Next, we remap race/ethnicity for more reabability.

In [262]:
race_map = {
    100: "White",
    200: "Black",
    300: "American Indian/Alaska Native",
    651: "Asian",
    652: "Native Hawaiian/Pacific Islander",

    801: "White & Black",
    802: "White & American Indian",
    803: "Black & American Indian",
    804: "White & Asian",
    805: "Black & Asian",
    806: "American Indian & Asian",
    807: "White, Black & American Indian",
    808: "White, Black & Asian",
    809: "White, American Indian & Asian",
    810: "Black, American Indian & Asian",
    811: "White & Pacific Islander",
    812: "Black & Pacific Islander",
    813: "Asian & Pacific Islander",
    814: "American Indian & Pacific Islander",
    815: "White, Black & Pacific Islander",
    816: "White, Asian & Pacific Islander",
    817: "Black, Asian & Pacific Islander",
    818: "White, American Indian & Pacific Islander",
    819: "Black, American Indian & Pacific Islander",
    820: "American Indian, Asian & Pacific Islander",
    830: "Other Multiple Race"
}

cps_income['RACE'] = cps_income['RACE'].map(race_map)
cps_income.head(50)


,YEAR,REGION,STATEFIP,COUNTY,METRO,ASECWT,AGE,SEX,RACE,OCC,UHRSWORK1,EDUC,SCHLCOLL,INCWAGE,OFFTOTVAL,POVERTY
0,2024,33,48,0,2,1898.77,80,Male,White,0,999,40,0,0,91488,23
1,2024,31,13,0,3,3187.62,64,Female,White,800,40,111,0,161000,294060,23
2,2024,12,36,0,4,2597.91,17,Female,White,0,999,60,1,0,55002,23
3,2024,21,17,17119,3,1810.53,25,Female,White,0,999,111,5,30000,43201,21
4,2024,31,37,37001,4,11338.34,45,Male,White,6600,45,71,5,55000,65000,23
5,2024,21,18,0,2,4448.76,54,Female,White,4230,999,123,5,12000,133000,23
6,2024,31,11,11001,2,419.87,41,Male,White,410,28,81,5,45429,45460,23
7,2024,33,40,0,3,3311.47,40,Male,White,4710,16,111,5,45000,143953,23
8,2024,42,6,6007,4,2336.91,77,Female,Asian,0,999,123,0,0,205186,23
9,2024,32,47,0,1,1177.87,13,Male,White,0,999,1,0,99999999,126963,23


Saved normalized/cleaned file...

#Creating ubi subset columns...

In [263]:
cps_income[['EDUC','SCHLCOLL','INCWAGE','AGE','SEX','RACE','OCC']].head()
UBI_subset = cps_income[['EDUC','SCHLCOLL','INCWAGE','AGE','SEX','RACE','OCC']]
UBI_subset.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   EDUC      1500 non-null   int64
 1   SCHLCOLL  1500 non-null   int64
 2   INCWAGE   1500 non-null   int64
 3   AGE       1500 non-null   int64
 4   SEX       1500 non-null   str  
 5   RACE      1500 non-null   str  
 6   OCC       1500 non-null   int64
dtypes: int64(5), str(2)
memory usage: 98.7 KB


In [264]:
#Cleaning ubi subset 'INCWAGE' column: removing place holders, with '0'
#Dropping nulls

UBI_subset['INCWAGE'] = UBI_subset['INCWAGE'].replace(99999999, np.nan)
UBI_subset = UBI_subset.dropna(subset=['INCWAGE'])
UBI_subset = UBI_subset[UBI_subset['INCWAGE'] > 0]
UBI_subset.head(100)

,EDUC,SCHLCOLL,INCWAGE,AGE,SEX,RACE,OCC
1,111,0,161000.0,64,Female,White,800
3,111,5,30000.0,25,Female,White,0
4,71,5,55000.0,45,Male,White,6600
5,123,5,12000.0,54,Female,White,4230
6,81,5,45429.0,41,Male,White,410
...,...,...,...,...,...,...,...
204,111,5,87000.0,38,Male,White,565
205,111,5,100000.0,32,Male,White,1430
209,111,0,98000.0,61,Male,White,960
210,73,0,3000.0,64,Male,White,0


In [265]:
#Creating a subset table to query...

In [266]:
UBI_subset = UBI_subset[['EDUC','SCHLCOLL','INCWAGE','AGE','SEX','RACE','OCC']]
UBI_subset.to_sql("ubi_subset", conn, if_exists="replace", index=False)



718

#SQL Queries...

In [267]:
df = pd.read_sql("""
SELECT *
FROM la_crime;
""", conn)

df

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,230514082,09/28/2023 12:00:00 AM,08/18/2023 12:00:00 AM,1800,5,Harbor,514,1,310,BURGLARY,...,IC,Invest Cont,310.0,NaN,NaN,None,1200 W PACIFIC COAST HY,NaN,33.7904,-118.2777
1,221311915,05/27/2022 12:00:00 AM,05/27/2022 12:00:00 AM,145,13,Newton,1373,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,998.0,NaN,None,56TH ST,CENTRAL ST,33.9916,-118.2564
2,231017635,12/27/2023 12:00:00 AM,12/26/2023 12:00:00 AM,800,10,West Valley,1067,2,740,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...",...,IC,Invest Cont,740.0,NaN,NaN,None,17400 VENTURA BL,NaN,34.1660,-118.5095
3,200211304,06/13/2020 12:00:00 AM,06/13/2020 12:00:00 AM,2300,2,Rampart,295,1,236,INTIMATE PARTNER - AGGRAVATED ASSAULT,...,AA,Adult Arrest,236.0,NaN,NaN,None,1300 CONSTANCE ST,NaN,34.0451,-118.2779
4,241206655,02/14/2024 12:00:00 AM,02/14/2024 12:00:00 AM,1600,12,77th Street,1208,2,930,CRIMINAL THREATS - NO WEAPON DISPLAYED,...,IC,Invest Cont,930.0,NaN,NaN,None,600 W VERNON AV,NaN,34.0038,-118.2842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,201111704,07/22/2020 12:00:00 AM,07/22/2020 12:00:00 AM,50,11,Northeast,1159,1,310,BURGLARY,...,AA,Adult Arrest,310.0,998.0,NaN,None,4500 N FIGUEROA ST,NaN,34.1009,-118.2036
1496,241405191,01/28/2024 12:00:00 AM,01/28/2024 12:00:00 AM,1210,14,Pacific,1443,2,745,VANDALISM - MISDEAMEANOR ($399 OR UNDER),...,IC,Invest Cont,745.0,NaN,NaN,None,VENICE BL,GRAND VIEW,33.9928,-118.4513
1497,211907685,04/15/2021 12:00:00 AM,04/15/2021 12:00:00 AM,1430,19,Mission,1974,2,624,BATTERY - SIMPLE ASSAULT,...,IC,Invest Cont,624.0,NaN,NaN,None,8900 KESTER AV,NaN,34.2318,-118.4575
1498,241807596,03/16/2024 12:00:00 AM,03/16/2024 12:00:00 AM,2055,18,Southeast,1838,1,761,BRANDISH WEAPON,...,IC,Invest Cont,761.0,NaN,NaN,None,10300 WILMINGTON AV,NaN,33.9432,-118.2391


In [268]:
pd.read_sql("""
SELECT "DATE OCC",
       "TIME OCC",
       LOCATION    
FROM la_crime
GROUP BY "DATE OCC"
ORDER BY "DATE OCC" DESC
LIMIT 7;
""", conn)

,DATE OCC,TIME OCC,LOCATION
0,12/31/2023 12:00:00 AM,1200,12500 VANOWEN ST
1,12/31/2021 12:00:00 AM,1835,9600 S WESTERN AV
2,12/30/2023 12:00:00 AM,1500,800 MORAGA DR
3,12/30/2022 12:00:00 AM,1500,13300 PAXTON ST
4,12/30/2021 12:00:00 AM,1730,2800 COLORADO BL
5,12/30/2020 12:00:00 AM,2130,4700 TACANA ST
6,12/29/2023 12:00:00 AM,900,4200 W PICO BL


In [269]:
pd.read_sql("""
SELECT
     LOCATION,
    "Weapon Desc", COUNT(*) AS occurrences
FROM 
    la_crime
WHERE
     "Weapon Desc" IS NOT NULL
GROUP BY 
    LOCATION, "Weapon Desc"
ORDER BY
     occurrences DESC
LIMIT 15;
""", conn)

,LOCATION,Weapon Desc,occurrences
0,100 S FIGUEROA ST,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
1,1800 W SLAUSON AV,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
2,4000 LAUREL CANYON BL,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
3,6TH ST,KNIFE WITH BLADE 6INCHES OR LESS,2
4,BROADWAY,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
5,CENTRAL AV,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",2
6,100 THE GROVE DR,VERBAL THREAT,1
7,100 E 11TH ST,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",1
8,100 S SERRANO AV,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",1
9,1000 BLAINE ST,SIMULATED GUN,1


Where is the MOST crime?

In [270]:
#LA crime count by location
pd.read_sql("""
SELECT
    LOCATION,
    COUNT(*) AS crime_count
FROM la_crime
GROUP BY LOCATION
ORDER BY crime_count DESC
LIMIT 1;
""", conn)

,LOCATION,crime_count
0,6TH ST,6


In [271]:
#Local crime count by zip code
pd.read_sql("""
SELECT
    ZIP_CODE,
    COUNT(*) AS crime_count
FROM local_crime
GROUP BY ZIP_CODE
ORDER BY crime_count DESC;
""", conn)

,zip_code,crime_count
0,40203,109
1,40219,103
2,40212,92
3,40214,79
4,40215,74
...,...,...
72,40177.0,1
73,40118.0,1
74,40023,1
75,40018,1


What time of day does the MOST crime occur?

In [272]:
pd.read_sql("""
SELECT
    LOCATION,
    "DATE OCC",
    "TIME OCC",
    COUNT(*) AS crime_frequency
FROM la_crime
GROUP BY LOCATION
ORDER BY crime_frequency DESC;
""", conn)

,LOCATION,DATE OCC,TIME OCC,crime_frequency
0,6TH ST,02/17/2020 12:00:00 AM,1038,6
1,MAIN ST,03/26/2021 12:00:00 AM,2015,4
2,BROADWAY,02/18/2021 12:00:00 AM,1700,4
3,8500 BEVERLY BL,11/22/2022 12:00:00 AM,1245,4
4,1800 W SLAUSON AV,09/27/2021 12:00:00 AM,50,4
...,...,...,...,...
1362,100 E 119TH ST,03/23/2022 12:00:00 AM,1,1
1363,100 BERTH,06/16/2021 12:00:00 AM,1930,1
1364,00 WORLD WAY,02/22/2020 12:00:00 AM,1130,1
1365,00 LMU DR,01/06/2022 12:00:00 AM,900,1


In [273]:
#Time of day most crimes occur: date_occured, offense_code_name, block_address, lmpd_division
pd.read_sql("""
SELECT 
    offense_code_name,
    block_address,
    lmpd_division,
    date_occurred
FROM local_crime
GROUP BY offense_code_name;
""", conn)

,offense_code_name,block_address,lmpd_division,date_occurred
0,ABANDONMENT OF MINOR 530.040 38200 90F,1700 BLOCK LAFAYETTE DR,4TH DIVISION,9/24/2023 9:53:00 PM
1,ACCIDENTAL SHOOTING (OTHER THAN HUNTING) ***.*...,NaN,1ST DIVISION,3/12/2023 9:00:00 PM
2,ANY NON CRIMINAL CHARGE NOT COVERED BY THESE C...,2700 BLOCK CRITTENDEN DR,4TH DIVISION,3/10/2023 3:32:00 AM
3,ASSAULT - 1ST DEGREE 508.010 13150 13A,SIXMILE ISLAND,8TH DIVISION,5/13/2023 5:20:00 AM
4,ASSAULT - 2ND DEGREE - DOMESTIC VIOLENCE 508.0...,1700 BLOCK BAIRD ST,1ST DIVISION,5/15/2023 1:00:00 PM
...,...,...,...,...
176,VIOLATION OF KENTUCKY EPO/DVO 403.763 02763 90F,400 BLOCK S 2ND ST,1ST DIVISION,6/15/2023 4:00:00 PM
177,VOYEURISM 531.090 01730 90H,300 BLOCK W MAIN ST,1ST DIVISION,6/28/2023 2:00:00 PM
178,WANTON ENDANGERMENT-1ST DEGREE 508.060 13201 13A,NaN,5TH DIVISION,6/13/2023 8:33:00 PM
179,WANTON ENDANGERMENT-1ST DEGREE-POLICE OFFICER ...,2100 BLOCK DIXIE HWY,2ND DIVISION,11/29/2023 12:49:00 AM


Converting 'date occurred' colum to time and date...

In [274]:
local_crime['date_occurred'] = pd.to_datetime(local_crime['date_occurred'])

C:\Users\bperf\AppData\Local\Temp\ipykernel_28824\1526250439.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  local_crime['date_occurred'] = pd.to_datetime(local_crime['date_occurred'])


Separatiang columns...

In [275]:
#Calculate columns for crime date, time, and hour from the 'date_occurred' column

local_crime['crime_date'] = local_crime['date_occurred'].dt.date

local_crime['crime_time'] = local_crime['date_occurred'].dt.time

local_crime['crime_hour'] = local_crime['date_occurred'].dt.hour


local_crime


,incident_number,date_reported,date_occurred,badge_id,offense_classification,offense_code_name,nibrs_code,nibrs_group_name,was_offense_completed,lmpd_division,lmpd_beat,location_category,block_address,city,zip_code,ObjectId,crime_date,crime_time,crime_hour
0,LMPD23111398,11/14/2023 8:44:00 PM,2023-11-14 20:45:00,5250.0,18 SHOPLIFTING,TBUT OR DISP SHOPLIFTING 514.030 24230 23C,23C,A,YES,1ST DIVISION,112,SERVICE/GAS STATION,3000 BLOCK W MUHAMMAD ALI BLVD,LOUISVILLE,40212,9195,2023-11-14,20:45:00,20
1,LMPD23105961,11/1/2023 5:08:00 AM,2023-11-01 05:08:00,5302.0,23 THEFT OTHER,TBUT OR DISP FIREARM 514.030 23100 23H,23H,A,YES,4TH DIVISION,423,RESIDENCE/HOME,1700 BLOCK VALLEY FORGE WAY,LOUISVILLE,40215,11201,2023-11-01,05:08:00,5
2,LMPD23076293,8/18/2023 7:29:00 PM,2023-08-18 17:30:00,8142.0,21 THEFT FR VEH,TBUT OR DISP CONTENTS FROM VEH 514.030 24140 23F,23F,A,YES,1ST DIVISION,123,PARKING/ DROP LOT/ GARAGE,100 BLOCK S 6TH ST,LOUISVILLE,40202,26195,2023-08-18,17:30:00,17
3,LMPD23028681,4/16/2023 3:40:00 AM,2023-04-16 03:40:00,5481.0,12 INTIMIDATION,INTIMIDATING A PARTICIPANT IN LEGAL PROCESS 52...,13C,A,YES,7TH DIVISION,711,RESIDENCE/HOME,9600 BLOCK HUDSON LN,LOUISVILLE,40291,50485,2023-04-16,03:40:00,3
4,LMPD23900829,7/5/2023 8:15:00 PM,2023-07-05 20:15:00,8375.0,23 LARCENY,"TBUT OR DISP ALL OTHERS $1,000 < $10,000 514.0...",23H,A,YES,2ND DIVISION,225,OTHER RESIDENCE (APARTMENT/CONDO),1600 BLOCK S 11TH ST,LOUISVILLE,40210,35539,2023-07-05,20:15:00,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,LMPD23082584,9/4/2023 3:26:00 PM,2023-09-04 03:00:00,8075.0,14 AUTO THEFT,"TBUT OR DISP AUTO $10,000 < $1,000,000 514.030...",240,A,YES,8TH DIVISION,824,PARKING/ DROP LOT/ GARAGE,2600 BLOCK SHINING WATER DR,LOUISVILLE,40299,23122,2023-09-04,03:00:00,3
1496,LMPD23055813,6/25/2023 10:40:00 PM,2023-06-25 22:40:00,4401.0,34 NARCOTICS,TRAFF IN CONTROLLED SUBSTANCE 1ST OFFENSE (HER...,35A,A,YES,3RD DIVISION,322,RESIDENCE/HOME,3100 BLOCK FORDHAVEN RD,LOUISVILLE,40214,37584,2023-06-25,22:40:00,22
1497,8023006888,1/24/2023 4:00:00 AM,2023-01-24 04:00:00,8142.0,21 THEFT FR VEH,TBUT OR DISP CONTENTS FROM VEH 514.030 24140 23F,23F,A,YES,3RD DIVISION,331,PARKING/ DROP LOT/ GARAGE,1600 BLOCK CLOVER ST,LOUISVILLE,40216.0,66612,2023-01-24,04:00:00,4
1498,LMPD23099753,10/16/2023 10:19:00 PM,2023-10-16 22:19:00,5556.0,11 SIMPLE ASSAULT,HARASSMENT - PHYSICAL CONTACT - NO INJURY 525....,13B,A,YES,4TH DIVISION,423,RESIDENCE/HOME,800 BLOCK BEECHER ST,LOUISVILLE,40215,14702,2023-10-16,22:19:00,22


In [276]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT objectid) AS unique_objectids
FROM local_crime;
""", conn)

,total_rows,unique_objectids
0,1500,1500


Time of day...

In [277]:
#AI assisted query: Time of day crimes occur:
local_crime['crime_hour'].value_counts().sort_index()

crime_hour
0      83
1      83
2      91
3      93
4     100
5      71
6      43
7      33
8      29
9      31
10     33
11     18
12     39
13     48
14     54
15     44
16     65
17     73
18     71
19     68
20     84
21     91
22     79
23     76
Name: count, dtype: int64

In [278]:
#AI SUGGESTED QUERY: #Time of day crimes occur: 

local_crime.groupby('crime_hour').size().reset_index(name='crime_count').sort_values(
    by='crime_hour',
    ascending=False
)

,crime_hour,crime_count
23,23,76
22,22,79
21,21,91
20,20,84
19,19,68
18,18,71
17,17,73
16,16,65
15,15,44
14,14,54


What crimes are occuring the most?

#AI assissted: refined search..

In [279]:
pd.read_sql("""
SELECT
    offense_classification,
    COUNT(*) AS crime_frequency
FROM local_crime
GROUP BY offense_classification
ORDER BY crime_frequency DESC;
""", conn)

,offense_classification,crime_frequency
0,56 ALL OTHER OFFENSES,170
1,24 VANDALISM,161
2,11 SIMPLE ASSAULT,147
3,14 AUTO THEFT,130
4,12 INTIMIDATION,101
5,23 THEFT OTHER,97
6,21 THEFT FR VEH,92
7,34 NARCOTICS,74
8,13 BURGLARY,73
9,9 AGGRAVATED ASSAULT,62


Areas with HIGHEST crime RATES?

In [280]:
pd.read_sql("""
SELECT
    zip_code,
    date_occurred,
    lmpd_division,
    offense_code_name,
    block_address,
    COUNT(*) AS crime_frequency
FROM local_crime
GROUP BY block_address
ORDER BY crime_frequency DESC;
""", conn)

,zip_code,date_occurred,lmpd_division,offense_code_name,block_address,crime_frequency
0,40205,6/13/2023 8:33:00 PM,5TH DIVISION,WANTON ENDANGERMENT-1ST DEGREE 508.060 13201 13A,NaN,34
1,40219,10/18/2023 9:54:00 PM,7TH DIVISION,TBUT OR DISP SHOPLIFTING 514.030 24230 23C,4800 BLOCK OUTER LOOP,8
2,40222,3/31/2023 6:15:00 PM,8TH DIVISION,TBUT OR DISP SHOPLIFTING 514.030 24230 23C,7900 BLOCK SHELBYVILLE RD,7
3,40202,7/13/2023 1:33:00 PM,1ST DIVISION,CRIMINAL MISCHIEF-3RD DEGREE 512.040 01403 290,400 BLOCK E MUHAMMAD ALI BLVD,7
4,40214,5/16/2023 10:52:00 PM,3RD DIVISION,RECOVERY OF STOLEN PROPERTY ***.*** 03016 90Z,100 BLOCK OUTER LOOP,7
...,...,...,...,...,...,...
1201,40206,9/20/2023 2:30:00 AM,5TH DIVISION,ASSAULT - 4TH DEGREE (DOMESTIC VIOLENCE) NO VI...,0 BLOCK HIGHWOOD DR,1
1202,40206,9/4/2023 7:30:00 PM,5TH DIVISION,TBUT OR DISP CONTENTS FROM VEH 514.030 24140 23F,0 BLOCK EASTOVER CT,1
1203,40203,9/2/2023 7:00:00 PM,4TH DIVISION,CRIMINAL MISCHIEF-2ND DEGREE 512.030 01402 290,0 BLOCK COLLEGE CT,1
1204,40214,4/16/2023 11:40:00 PM,3RD DIVISION,DOMESTIC ABUSE DUTIES OF LAW ENFORCEMENT 403.7...,0 BLOCK CALUMET DR,1


In [281]:
pd.read_sql("""
SELECT
    zip_code,
    COUNT(*) AS crime_frequency
FROM local_crime
GROUP BY zip_code
ORDER BY crime_frequency DESC;
""", conn)

,zip_code,crime_frequency
0,40203,109
1,40219,103
2,40212,92
3,40214,79
4,40215,74
...,...,...
72,40177.0,1
73,40118.0,1
74,40023,1
75,40018,1


In [282]:
pd.read_sql("""
SELECT
    lmpd_division,
    COUNT(*) AS crime_frequency
FROM local_crime
GROUP BY lmpd_division
ORDER BY crime_frequency DESC;
""", conn)

,lmpd_division,crime_frequency
0,4TH DIVISION,289
1,1ST DIVISION,214
2,2ND DIVISION,204
3,3RD DIVISION,198
4,6TH DIVISION,189
5,7TH DIVISION,183
6,8TH DIVISION,106
7,5TH DIVISION,100
8,NaN,7
9,ST MATTHEWS,4


What areas are experiecing a specific crime:  burgalries, theft, robbery...

In [283]:
pd.read_sql("""
SELECT
    zip_code,
    COUNT(*) AS burglary_count
FROM local_crime
WHERE offense_code_name LIKE '%BURGLARY%'
GROUP BY zip_code
ORDER BY burglary_count DESC;
""", conn)

,zip_code,burglary_count
0,40211,7
1,40212,6
2,40218,5
3,40229,4
4,40219,4
5,40215,4
6,40272,3
7,40258,3
8,40241,3
9,40216,3


In [284]:
pd.read_sql("""
SELECT
    zip_code,
    COUNT(*) AS WANTON_ENDANGERMENT_count
FROM local_crime
WHERE offense_code_name LIKE '%WANTON_ENDANGERMENT%'
GROUP BY zip_code
ORDER BY WANTON_ENDANGERMENT_count DESC;
""", conn)
#WANTON_ENDANGERMENT

,zip_code,WANTON_ENDANGERMENT_count
0,40210,4
1,40218,3
2,40216,3
3,40212,3
4,40272,2
5,40219,2
6,40215,2
7,40213,2
8,40208,2
9,40204,2


In [285]:
pd.read_sql("""
SELECT
    zip_code,
    COUNT(*) AS THEFT_count
FROM local_crime
WHERE offense_code_name LIKE '%THEFT%'
GROUP BY zip_code
ORDER BY THEFT_count DESC;
""", conn)

,zip_code,THEFT_count
0,40213,9
1,40203,9
2,40214,8
3,40219,6
4,40208,6
5,40272,5
6,40218,5
7,40299,4
8,40258,4
9,40212,4


In [286]:
pd.read_sql("""
SELECT
    zip_code,
    COUNT(*) AS DOMESTIC_count
FROM local_crime
WHERE offense_code_name LIKE '%DOMESTIC%'
GROUP BY zip_code
ORDER BY DOMESTIC_count DESC;
""", conn)

,zip_code,DOMESTIC_count
0,40219,18
1,40212,18
2,40203,16
3,40214,14
4,40272,10
5,40210,9
6,40291,8
7,40215,8
8,40216,7
9,40208,7


In [287]:
pd.read_sql("""
SELECT
    zip_code,
    COUNT(*) AS ROBBERY_count
FROM local_crime
WHERE offense_code_name LIKE '%ROBBERY%'
GROUP BY zip_code
ORDER BY ROBBERY_count DESC;
""", conn)

,zip_code,ROBBERY_count
0,40203,3
1,40219,2
2,40215,2
3,40219.0,1
4,40217,1
5,40211,1
6,40210,1
7,40209,1
8,40206,1
9,40204,1


What areas are experiencing the lowest crime rates?

In [294]:
pd.read_sql("""
SELECT
    zip_code,
    COUNT(*) AS crime_frequency
FROM local_crime
GROUP BY zip_code
ORDER BY crime_frequency asc;
""", conn)

,zip_code,crime_frequency
0,2015,1
1,40018,1
2,40023,1
3,40118.0,1
4,40177.0,1
...,...,...
72,40215,74
73,40214,79
74,40212,92
75,40219,103


Average income:  by sex, race, age, education.....

In [288]:
#avg icome by sex
avg_income = pd.read_sql(
    """
    SELECT
        SEX,
    ROUND(AVG(INCWAGE),2) AS avg_income
    FROM ubi_subset
    GROUP BY SEX;
    """,
    conn
)

avg_income

,SEX,avg_income
0,Female,53390.35
1,Male,84378.41


In [289]:
#avg icome by race
avg_income = pd.read_sql(
    """
    SELECT
        RACE,
    ROUND(AVG(INCWAGE),2) AS avg_income
    FROM ubi_subset
    GROUP BY RACE
    ORDER BY AVG_INCOME DESC;
    """,
    conn
)

avg_income

,RACE,avg_income
0,Asian,97443.24
1,Black & American Indian,87331.50
2,White,70081.86
3,White & Asian,63509.50
4,Black,56972.78
5,Asian & Pacific Islander,52000.00
6,American Indian/Alaska Native,49527.27
7,White & Black,49333.33
8,White & American Indian,48048.57
9,Black & Pacific Islander,35000.00


In [290]:
#avg icome by age
avg_income = pd.read_sql(
    """
    SELECT
        AGE,
    ROUND(AVG(INCWAGE),2) AS avg_income
    FROM ubi_subset
    GROUP BY AGE;
    """,
    conn
)

avg_income

,AGE,avg_income
0,15,15200.00
1,16,11875.00
2,17,7036.67
3,18,11500.00
4,19,27544.44
...,...,...
61,76,18000.00
62,77,13500.00
63,78,11000.00
64,80,73600.00


In [291]:
#avg icome by education
avg_income = pd.read_sql(
    """
    SELECT
        EDUC,
    ROUND(AVG(INCWAGE),2) AS avg_income
    FROM ubi_subset
    GROUP BY EDUC
    ORDER BY EDUC ASC;
    """,
    conn
)

avg_income

,EDUC,avg_income
0,10,42066.67
1,20,14250.00
2,30,208333.17
3,40,25688.89
4,50,26055.56
5,60,27741.00
6,71,35333.33
7,73,47812.70
8,81,53897.41
9,91,59306.24


In [ ]:
#zip code, avg income, and race, sex, age, education, occupation#

Correlation Heatmap

Geographic Crime Map

In [ ]:
# import folium

# m = folium.Map(location=[39,-95], zoom_start=11)

# for _, row in la_crime.iterrows():
#     folium.CircleMarker(
#         [row['LAT'], row['LON']],
#         radius=3,
#         color="red"
#     ).add_to(m)

# m